Gobierno de Datos.

Imagina que estás construyendo un asistente virtual para el equipo de datos de un banco. Queremos que la IA sea capaz de buscar información sobre los activos de datos de la empresa simulando una conexión a un Catálogo de Datos (como DataHub).

Asistente de Catálogo de Metadatos
Este ejemplo crea una herramienta que busca si una tabla existe en el catálogo del banco, quién es su responsable (Data Owner) y su nivel de calidad de datos.

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

In [2]:
# 1. Cargamos la clave de API
load_dotenv()

True

In [3]:
# 2. Definimos la herramienta avanzada
# Nota cómo los Type Hints y el Docstring le explican a la IA exactamente qué hace la función.
@tool
def consultar_catalogo_datos(nombre_tabla: str, dominio_datos: str = "general") -> str:
    """
    Consulta el catálogo de metadatos de la empresa para obtener información sobre un activo de datos.
    
    Args:
        nombre_tabla: El nombre de la tabla de base de datos a buscar (ej. 'clientes', 'transacciones', 'riesgo_crediticio').
        dominio_datos: El dominio de negocio al que pertenece (ej. 'riesgos', 'comercial', 'operaciones').
    """
    
    # Simulamos una base de datos de nuestro catálogo (como si fuera DataHub o un gestor de metadatos)
    catalogo_simulado = {
        "riesgo_crediticio": {
            "descripcion": "Historial de evaluación crediticia de clientes minoristas.",
            "data_owner": "Ana Torres (Gerencia de Riesgos)",
            "frecuencia_actualizacion": "Diaria",
            "calidad_datos_score": "98%",
            "cumple_normativa": True
        },
        "transacciones": {
            "descripcion": "Registro transaccional de tarjetas de crédito.",
            "data_owner": "Carlos Silva (Operaciones)",
            "frecuencia_actualizacion": "Tiempo Real",
            "calidad_datos_score": "85%",
            "cumple_normativa": True
        }
    }
    
    # Buscamos la tabla, ignorando mayúsculas/minúsculas
    tabla_limpia = nombre_tabla.lower()
    
    if tabla_limpia in catalogo_simulado:
        resultado = catalogo_simulado[tabla_limpia]
        resultado["tabla"] = tabla_limpia
        resultado["dominio_registrado"] = dominio_datos
        return json.dumps(resultado)
    else:
        return json.dumps({"error": f"La tabla '{nombre_tabla}' no se encontró en el catálogo."})

In [4]:
# 3. Inicializamos el modelo (usando la versión que te funcionó perfectamente)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [5]:
# 4. Enlazamos la herramienta al modelo
llm_con_catalogo = llm.bind_tools([consultar_catalogo_datos])

In [6]:
# 5. Simulamos la consulta de un usuario de negocio
mensaje_usuario = "Hola, necesito saber quién es el dueño de la tabla de riesgo_crediticio en el dominio de riesgos y cuál es su score de calidad."
print(f"Usuario: {mensaje_usuario}")

Usuario: Hola, necesito saber quién es el dueño de la tabla de riesgo_crediticio en el dominio de riesgos y cuál es su score de calidad.


In [7]:
# Ejecutamos el modelo
respuesta = llm_con_catalogo.invoke(mensaje_usuario)

In [8]:
# 6. Procesamos la decisión de la IA
print("\n--- Análisis de la IA ---")
if respuesta.tool_calls:
    llamada = respuesta.tool_calls[0]
    print(f"Herramienta seleccionada: {llamada['name']}")
    
    # Extraemos los argumentos que la IA identificó inteligentemente del texto del usuario
    argumentos = llamada["args"]
    print(f"Argumentos extraídos: {argumentos}")
    
    # 7. Ejecutamos la función de Python
    resultado_metadatos = consultar_catalogo_datos.invoke(argumentos)
    print("\n--- Resultado de la Ejecución (Lo que devuelve el sistema) ---")
    print(resultado_metadatos)
    
else:
    print("La IA decidió no usar ninguna herramienta y respondió directamente:")
    print(respuesta.content)


--- Análisis de la IA ---
Herramienta seleccionada: consultar_catalogo_datos
Argumentos extraídos: {'nombre_tabla': 'riesgo_crediticio', 'dominio_datos': 'riesgos'}

--- Resultado de la Ejecución (Lo que devuelve el sistema) ---
{"descripcion": "Historial de evaluaci\u00f3n crediticia de clientes minoristas.", "data_owner": "Ana Torres (Gerencia de Riesgos)", "frecuencia_actualizacion": "Diaria", "calidad_datos_score": "98%", "cumple_normativa": true, "tabla": "riesgo_crediticio", "dominio_registrado": "riesgos"}
